# Silver: limpieza y tipado
Lee las 7 tablas de Bronze, aplica las reglas de limpieza aprobadas y guarda el resultado en `dunnhumby.silver`. Las filas rechazadas van a `transaction_data_cuarentena`.

In [0]:
# SECCIÓN 0 · Funciones comunes (reglas 1 y 2: nombres en minúsculas, tipos y texto sin espacios)
from pyspark.sql import functions as F

def construir_silver(nombre, tipos):
    """Lee la tabla de Bronze y aplica las reglas comunes: nombres en minúsculas, tipos y texto sin espacios."""
    df = spark.table(f"dunnhumby.bronze.{nombre}").drop("_archivo_origen", "_fecha_carga")
    df = df.toDF(*[c.lower() for c in df.columns])   # regla 1: Bronze mezcla minúsculas y mayúsculas
    for c in df.columns:
        if c in tipos:
            # try_cast devuelve NULL si el texto no es convertible, en lugar de lanzar error
            df = df.withColumn(c, F.expr(f"try_cast(`{c}` AS {tipos[c]})"))
        else:
            df = df.withColumn(c, F.trim(F.col(c)))  # regla 2: texto sin espacios sobrantes
    return df.withColumn("_fecha_proceso", F.current_timestamp())


def guardar_silver(df, nombre, tipos):
    """Verifica que la conversión de tipos no haya creado NULL y guarda la tabla en silver."""
    nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in tipos]).first()
    problemas = {c: nulos[c] for c in tipos if nulos[c]}
    assert not problemas, f"Valores no convertibles en {nombre}: {problemas}"

    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")   # permite re-ejecutar aunque cambie el esquema
       .saveAsTable(f"dunnhumby.silver.{nombre}"))

    n_bronze = spark.table(f"dunnhumby.bronze.{nombre}").count()
    n_silver = spark.table(f"dunnhumby.silver.{nombre}").count()
    print(f"{nombre}: bronze {n_bronze:,} filas -> silver {n_silver:,} filas")

## 1. Tablas simples
Solo necesitan tipos y limpieza de texto: `campaign_desc`, `campaign_table` y `coupon_redempt`.

In [0]:
# SECCIÓN 1 · campaign_desc, campaign_table, coupon_redempt · solo tipos
TABLAS_SIMPLES = {
    "campaign_desc":  {"campaign": "int", "start_day": "int", "end_day": "int"},
    "campaign_table": {"household_key": "int", "campaign": "int"},
    "coupon_redempt": {"household_key": "int", "day": "int", "coupon_upc": "bigint", "campaign": "int"},
}

for nombre, tipos in TABLAS_SIMPLES.items():
    guardar_silver(construir_silver(nombre, tipos), nombre, tipos)

In [0]:
# SECCIÓN 1 · Verificación de tipos de las tablas simples
# Esperado: columnas en minúsculas, identificadores integer/long y _fecha_proceso como timestamp
for nombre in TABLAS_SIMPLES:
    print(nombre)
    spark.table(f"dunnhumby.silver.{nombre}").printSchema()

## 2. coupon y hh_demographic
Tablas con reglas propias: duplicados en `coupon` y valores de relleno en `hh_demographic`.

In [0]:
# SECCIÓN 2 · coupon · regla 12: eliminar duplicados exactos
tipos_coupon = {"coupon_upc": "bigint", "product_id": "bigint", "campaign": "int"}
df = construir_silver("coupon", tipos_coupon)

# Regla 12: se eliminan las filas idénticas. _fecha_proceso se excluye de la comparación
columnas = [c for c in df.columns if c != "_fecha_proceso"]
df = df.dropDuplicates(columnas)

guardar_silver(df, "coupon", tipos_coupon)

In [0]:
# SECCIÓN 2 · hh_demographic · reglas 10 y 11: desconocidos a NULL y orden de ingreso
tipos_hh = {"household_key": "int"}
df = construir_silver("hh_demographic", tipos_hh)

# Regla 10: los valores de relleno pasan a NULL. kid_category_desc no se toca: "None/Unknown" mezcla dos significados
for c in ["hh_comp_desc", "homeowner_desc"]:
    df = df.withColumn(c, F.when(F.col(c) == "Unknown", None).otherwise(F.col(c)))
df = df.withColumn("marital_status_code",
                   F.when(F.col("marital_status_code") == "U", None).otherwise(F.col("marital_status_code")))

# Regla 11: como texto, "15-24K" ordena después de "125-149K"; income_orden da el orden real
ORDEN_INGRESO = ["Under 15K", "15-24K", "25-34K", "35-49K", "50-74K", "75-99K",
                 "100-124K", "125-149K", "150-174K", "175-199K", "200-249K", "250K+"]
mapa = F.create_map(*[x for i, r in enumerate(ORDEN_INGRESO, 1) for x in (F.lit(r), F.lit(i))])
df = df.withColumn("income_orden", mapa[F.col("income_desc")])

# Todos los rangos de Bronze deben estar en la lista; si no, income_orden quedaría NULL
assert df.filter("income_orden IS NULL").count() == 0, "Hay rangos de ingreso fuera de la lista"

guardar_silver(df, "hh_demographic", tipos_hh)

In [0]:
# SECCIÓN 2 · Verificación de coupon y hh_demographic
# Esperado: coupon con 119,384 filas y 0 duplicadas; nulos de hh_comp 73, homeowner 233, marital 344; kid_none_unknown 558
c = spark.table("dunnhumby.silver.coupon").drop("_fecha_proceso")
print(f"coupon: {c.count():,} filas, {c.count() - c.distinct().count():,} duplicadas")

hh = spark.table("dunnhumby.silver.hh_demographic")
display(hh.select(
    F.sum(F.col("hh_comp_desc").isNull().cast("int")).alias("hh_comp_nulos"),
    F.sum(F.col("homeowner_desc").isNull().cast("int")).alias("homeowner_nulos"),
    F.sum(F.col("marital_status_code").isNull().cast("int")).alias("marital_nulos"),
    F.sum((F.col("kid_category_desc") == "None/Unknown").cast("int")).alias("kid_none_unknown"),
))
display(hh.groupBy("income_orden", "income_desc").count().orderBy("income_orden"))

## 3. product
Catálogo de productos: vacíos a `NULL`, productos sin clasificar y marca de no mercadería.

In [0]:
# SECCIÓN 3 · product · reglas 7, 8 y 9
tipos_product = {"product_id": "bigint", "manufacturer": "int"}
df = construir_silver("product", tipos_product)

# Regla 7: un tamaño vacío no es un valor, es la ausencia de dato
df = df.withColumn("curr_size_of_product",
                   F.when(F.col("curr_size_of_product") == "", None).otherwise(F.col("curr_size_of_product")))

# Regla 8: los 15 productos sin clasificar se conservan, con una etiqueta explícita
# La condición se guarda en una columna antes del ciclo: si se evaluara dentro, department ya estaría reemplazado
df = df.withColumn("_sin_clasificar", F.col("department").isNull() | (F.col("department") == ""))
for c in ["department", "commodity_desc", "sub_commodity_desc"]:
    df = df.withColumn(c, F.when(F.col("_sin_clasificar"), "SIN CLASIFICAR").otherwise(F.col(c)))
df = df.drop("_sin_clasificar")

# Regla 9: su cantidad no es comparable con la de la mercadería (la gasolina se mide en otra unidad)
NO_MERCADERIA = ["KIOSK-GAS", "MISC SALES TRAN", "MISC. TRANS."]
df = df.withColumn("es_no_mercaderia", F.col("department").isin(NO_MERCADERIA))

guardar_silver(df, "product", tipos_product)

In [0]:
# SECCIÓN 3 · Verificación de product
# Esperado: size_nulos 30,607; 15 sin clasificar en las tres columnas; no_mercaderia 594 (KIOSK-GAS 16, MISC SALES TRAN 88, MISC. TRANS. 490)
pr = spark.table("dunnhumby.silver.product")

display(pr.select(
    F.sum(F.col("curr_size_of_product").isNull().cast("int")).alias("size_nulos"),
    F.sum((F.col("department") == "SIN CLASIFICAR").cast("int")).alias("dept_sin_clasificar"),
    F.sum((F.col("commodity_desc") == "SIN CLASIFICAR").cast("int")).alias("commodity_sin_clasificar"),
    F.sum((F.col("sub_commodity_desc") == "SIN CLASIFICAR").cast("int")).alias("sub_sin_clasificar"),
    F.sum(F.col("es_no_mercaderia").cast("int")).alias("no_mercaderia"),
))
display(pr.filter("es_no_mercaderia").groupBy("department").count().orderBy("department"))

## 4. transaction_data y cuarentena
Tabla central: importes redondeados, hora derivada y separación de las líneas sin unidades.

In [0]:
# SECCIÓN 4 · transaction_data · reglas 3, 4, 5 y 6 (las líneas rechazadas van a cuarentena)
tipos_tx = {
    "household_key": "int", "basket_id": "bigint", "day": "int", "product_id": "bigint",
    "quantity": "int", "sales_value": "decimal(10,2)", "store_id": "int",
    "retail_disc": "decimal(10,2)", "trans_time": "int", "week_no": "int",
    "coupon_disc": "decimal(10,2)", "coupon_match_disc": "decimal(10,2)",
}

# Regla 3: pasar a decimal(10,2) redondea los importes y elimina el ruido de punto flotante (2.22E-16)
df = construir_silver("transaction_data", tipos_tx)

# Se comprueba que ningún valor haya quedado en NULL antes de separar las filas; si no, alguna fila no caería en ninguna tabla
nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in tipos_tx]).first()
assert not any(nulos[c] for c in tipos_tx), "Valores no convertibles en transaction_data"

# Regla 6: trans_time viene como HHMM sin ceros a la izquierda (45 significa 00:45)
assert df.filter((F.col("trans_time") % 100) >= 60).count() == 0, "Hay minutos inválidos en trans_time"
df = df.withColumn("hora", F.floor(F.col("trans_time") / 100).cast("int"))

# Regla 4: una línea sin unidades no es una compra; va a cuarentena con su motivo
cantidad_cero = F.col("quantity") == 0
cuarentena = df.filter(cantidad_cero).withColumn(
    "motivo",
    F.when(F.col("sales_value") == 0, "cantidad y venta en cero")
     .otherwise("cantidad cero con venta positiva"),
)
principal = df.filter(~cantidad_cero)   # la regla 5 se cumple sola: las líneas con unidades y venta 0 se quedan aquí

(cuarentena.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("dunnhumby.silver.transaction_data_cuarentena"))
guardar_silver(principal, "transaction_data", tipos_tx)

# Cuadre: toda fila de Bronze debe estar en una de las dos tablas
n_b = spark.table("dunnhumby.bronze.transaction_data").count()
n_s = spark.table("dunnhumby.silver.transaction_data").count()
n_c = spark.table("dunnhumby.silver.transaction_data_cuarentena").count()
assert n_b == n_s + n_c, "Las filas no cuadran con Bronze"
print(f"Cuadre: {n_b:,} = {n_s:,} (silver) + {n_c:,} (cuarentena)")

In [0]:
# SECCIÓN 4 · Verificación de transaction_data y su cuarentena
# Esperado: cuarentena de 14,466 filas en total (14,428 + 38); quantity_min 1; hora de 0 a 23
display(
    spark.table("dunnhumby.silver.transaction_data_cuarentena")
    .groupBy("motivo")
    .agg(F.count("*").alias("filas"), F.sum("sales_value").alias("venta_total"))
)

tx = spark.table("dunnhumby.silver.transaction_data")
display(tx.select(
    F.min("quantity").alias("quantity_min"),
    F.sum((F.col("sales_value") == 0).cast("int")).alias("venta_cero"),
    F.sum((F.col("retail_disc") > 0).cast("int")).alias("retail_disc_positivo"),
    F.min("hora").alias("hora_min"),
    F.max("hora").alias("hora_max"),
))